#HW3: RAG and Data Extraction
## Part A: Retrieval-augmented generation
### Executed result summary (August 13, 2026):

Using gpt-5.6-terra, the Few-shot - open embeddings method (using sentence-transformers/all-mpnet-base-v2 for embeddings) achieved the highest performance with 69.3% accuracy and 0.706 macro F1. The Few-shot - OpenAI embeddings (text-embedding-3-small) method followed with 66.7% accuracy and 0.675 macro F1.

For KNN approaches, KNN - OpenAI embeddings (text-embedding-3-small) reached 60.8% accuracy / 0.596 macro F1, while KNN - open embeddings reached 58.2% accuracy / 0.543 macro F1.

The Zero-shot method achieved 54.9% accuracy / 0.456 macro F1.

For this dataset, the Few-shot - open embeddings method provided the best overall performance, demonstrating the effectiveness of RAG. When minimizing ongoing API cost, KNN - open embeddings offers a lower-cost alternative with slightly reduced performance. The current project key can access OpenAI embedding models, as evidenced by the successful execution and results of the OpenAI-based classifiers

In [1]:
#load required packages
from pathlib import Path
from collections import Counter
import base64
import json
import os
import time

!pip install pymupdf -q
!pip install langchain-chroma -q
!pip install langchain-huggingface -q
!pip install langchain-openai -q
import fitz
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings
from openai import OpenAI
from sklearn.metrics import accuracy_score, classification_report, f1_score

ROOT = Path.cwd()
load_dotenv(ROOT / ".env")
load_dotenv(ROOT.parent / ".env")

TEXT_MODEL = os.getenv("OPENAI_TEXT_MODEL", "gpt-5.6-terra")
OPENAI_EMBED_MODEL = os.getenv("OPENAI_EMBED_MODEL", "text-embedding-3-small")
OPEN_EMBED_MODEL = os.getenv("OPEN_EMBED_MODEL", "sentence-transformers/all-mpnet-base-v2")
HF_HOME = ROOT / ".cache" / "huggingface"
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY was not found in .env"
client = OpenAI()

print({
    "text_model": TEXT_MODEL,
    "openai_embedding_model": OPENAI_EMBED_MODEL,
    "open_embedding_model": OPEN_EMBED_MODEL,
    "api_key_present": True,
})

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 82.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 101.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/

In [2]:
#load and inspect data
def find_ticket_file(stem):
    for suffix, separator in [(".tsv", "\t"), (".csv", ",")]:
        path = ROOT / f"{stem}{suffix}"
        if path.exists():
            return path, separator
    raise FileNotFoundError(f"Could not find {stem}.tsv or {stem}.csv")


train_path, train_sep = find_ticket_file("ticket_train")
test_path, test_sep = find_ticket_file("ticket_test")
train_df = pd.read_csv(train_path, sep=train_sep)[["text", "label"]].dropna().reset_index(drop=True)
test_df = pd.read_csv(test_path, sep=test_sep)[["text", "label"]].dropna().reset_index(drop=True)

LABELS = sorted(train_df["label"].unique().tolist())
print(f"Training rows: {len(train_df)}")
print(f"Testing rows: {len(test_df)}")
print(f"Labels ({len(LABELS)}): {LABELS}")
display(train_df.head(3))

Training rows: 150
Testing rows: 153
Labels (5): ['Billing and Payments', 'Customer Service', 'IT Support', 'Product Support', 'Technical Support']


,text,label
0,"Dear Tech Online Store Support,\n\nI am writin...",Product Support
1,"Hello, I am experiencing frequent disconnectio...",Technical Support
2,"Dear Tech Online Store Customer Support,\n\nI ...",Technical Support


In [3]:
#check which OpenAI embedding model this key can access¶
EMBEDDING_CANDIDATES = list(dict.fromkeys([
    OPENAI_EMBED_MODEL,
    "text-embedding-3-small",
    "text-embedding-3-large",
    "text-embedding-ada-002",
]))

embedding_access_rows = []
for model_name in EMBEDDING_CANDIDATES:
    try:
        response = client.embeddings.create(model=model_name, input=["availability test"])
        embedding_access_rows.append({
            "model": model_name,
            "available": True,
            "dimensions": len(response.data[0].embedding),
            "result": "available",
        })
    except Exception as exc:
        embedding_access_rows.append({
            "model": model_name,
            "available": False,
            "dimensions": pd.NA,
            "result": f"{type(exc).__name__} (HTTP {getattr(exc, 'status_code', 'unknown')})",
        })

embedding_access = pd.DataFrame(embedding_access_rows)
AVAILABLE_OPENAI_EMBED_MODELS = embedding_access.loc[
    embedding_access["available"], "model"
].tolist()
display(embedding_access)

,model,available,dimensions,result
0,text-embedding-3-small,True,1536,available
1,text-embedding-3-large,True,3072,available
2,text-embedding-ada-002,True,1536,available


In [4]:
#build two Chroma vector stores
documents = [
    Document(page_content=row.text, metadata={"label": row.label, "ticket_id": int(i)})
    for i, row in train_df.iterrows()
]
document_ids = [f"ticket_{i}" for i in range(len(documents))]

if AVAILABLE_OPENAI_EMBED_MODELS:
    OPENAI_EMBED_MODEL = AVAILABLE_OPENAI_EMBED_MODELS[0]
    openai_embeddings = OpenAIEmbeddings(model=OPENAI_EMBED_MODEL)
    OPENAI_EMBEDDINGS_LIVE = True
    OPENAI_STORE_LABEL = f"OpenAI embeddings ({OPENAI_EMBED_MODEL})"
    openai_collection_name = "hw3_tickets_" + OPENAI_EMBED_MODEL.replace("-", "_")
    openai_persist_directory = ROOT / ("chroma_" + openai_collection_name)
else:
    OPENAI_EMBEDDINGS_LIVE = False
    OPENAI_STORE_LABEL = "MiniLM fallback (OpenAI embeddings unavailable)"
    openai_collection_name = "hw3_tickets_minilm_fallback"
    openai_persist_directory = ROOT / "chroma_hw3_minilm_fallback"
    print("No documented OpenAI embedding model is available. Using a labeled MiniLM fallback.")
    openai_embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        cache_folder=str(HF_HOME),
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )
open_embeddings = HuggingFaceEmbeddings(
    model_name=OPEN_EMBED_MODEL,
    cache_folder=str(HF_HOME),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

openai_store = Chroma(
    collection_name=openai_collection_name,
    embedding_function=openai_embeddings,
    persist_directory=str(openai_persist_directory),
)
open_store = Chroma(
    collection_name="hw3_tickets_open",
    embedding_function=open_embeddings,
    persist_directory=str(ROOT / "chroma_hw3_open"),
)

openai_store.add_documents(documents, ids=document_ids)
open_store.add_documents(documents, ids=document_ids)
print(f"Added {len(documents)} training tickets to each Chroma store.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Added 150 training tickets to each Chroma store.


In [5]:
#KNN classifiers
def classify_ticket_by_knn(ticket, vector_store, k=3):
    neighbors = vector_store.similarity_search(ticket, k=k)
    neighbor_labels = [doc.metadata["label"] for doc in neighbors]
    counts = Counter(neighbor_labels)
    best_count = max(counts.values())
    return sorted(label for label, count in counts.items() if count == best_count)[0]


pred_knn_openai = [classify_ticket_by_knn(text, openai_store) for text in test_df["text"]]
pred_knn_open = [classify_ticket_by_knn(text, open_store) for text in test_df["text"]]

print(f"KNN - {OPENAI_STORE_LABEL}")
print(classification_report(test_df["label"], pred_knn_openai, labels=LABELS, zero_division=0))
print("KNN - open embeddings")
print(classification_report(test_df["label"], pred_knn_open, labels=LABELS, zero_division=0))

KNN - OpenAI embeddings (text-embedding-3-small)
                      precision    recall  f1-score   support

Billing and Payments       0.86      0.92      0.89        13
    Customer Service       0.46      0.50      0.48        24
          IT Support       0.39      0.37      0.38        19
     Product Support       0.44      0.70      0.54        23
   Technical Support       0.78      0.62      0.69        74

            accuracy                           0.61       153
           macro avg       0.59      0.62      0.60       153
        weighted avg       0.64      0.61      0.61       153

KNN - open embeddings
                      precision    recall  f1-score   support

Billing and Payments       0.79      0.85      0.81        13
    Customer Service       0.52      0.62      0.57        24
          IT Support       0.22      0.11      0.14        19
     Product Support       0.42      0.74      0.54        23
   Technical Support       0.72      0.59      0.65      

In [6]:
#Zero-shot classifier
LABEL_SCHEMA = {
    "type": "json_schema",
    "name": "ticket_classification",
    "schema": {
        "type": "object",
        "properties": {"label": {"type": "string", "enum": LABELS}},
        "required": ["label"],
        "additionalProperties": False,
    },
    "strict": True,
}


def classify_ticket_zero_shot(ticket):
    response = client.responses.create(
        model=TEXT_MODEL,
        reasoning={"effort": "none"},
        instructions=(
            "Classify the insurance support ticket into exactly one allowed label. "
            "Return only the structured result."
        ),
        input=f"Allowed labels: {LABELS}\n\nTicket: {ticket}",
        text={"format": LABEL_SCHEMA},
        store=False,
    )
    return json.loads(response.output_text)["label"]


def predict_with_cache(name, classify_function):
    output_dir = ROOT / "outputs" / "part_a"
    output_dir.mkdir(parents=True, exist_ok=True)
    cache_path = output_dir / f"{name}.csv"
    predictions = []
    if cache_path.exists():
        cached = pd.read_csv(cache_path)
        if len(cached) == len(test_df):
            print(f"Using cached predictions: {cache_path}")
            return cached["prediction"].tolist()
        predictions = cached["prediction"].dropna().tolist()
        print(f"Resuming {name} after {len(predictions)} cached predictions.")

    for row_index in range(len(predictions), len(test_df)):
        ticket = test_df.iloc[row_index]["text"]
        predictions.append(classify_function(ticket))
        completed = len(predictions)
        pd.DataFrame({
            "text": test_df.iloc[:completed]["text"],
            "label": test_df.iloc[:completed]["label"],
            "prediction": predictions,
        }).to_csv(cache_path, index=False)
        if completed % 10 == 0 or completed == len(test_df):
            print(f"{name}: {completed}/{len(test_df)}")
    return predictions


pred_zero_shot = predict_with_cache("zero_shot", classify_ticket_zero_shot)
print(classification_report(test_df["label"], pred_zero_shot, labels=LABELS, zero_division=0))

zero_shot: 10/153
zero_shot: 20/153
zero_shot: 30/153
zero_shot: 40/153
zero_shot: 50/153
zero_shot: 60/153
zero_shot: 70/153
zero_shot: 80/153
zero_shot: 90/153
zero_shot: 100/153
zero_shot: 110/153
zero_shot: 120/153
zero_shot: 130/153
zero_shot: 140/153
zero_shot: 150/153
zero_shot: 153/153
                      precision    recall  f1-score   support

Billing and Payments       0.76      1.00      0.87        13
    Customer Service       0.50      0.08      0.14        24
          IT Support       0.17      0.21      0.19        19
     Product Support       0.58      0.30      0.40        23
   Technical Support       0.60      0.78      0.68        74

            accuracy                           0.55       153
           macro avg       0.52      0.48      0.46       153
        weighted avg       0.54      0.55      0.51       153



In [7]:
#Retrieved few-shot classifiers
def classify_ticket_with_rag(ticket, vector_store, k=3):
    examples = vector_store.similarity_search(ticket, k=k)
    demonstrations = "\n\n".join(
        f"Ticket: {doc.page_content}\nLabel: {doc.metadata['label']}"
        for doc in examples
    )
    prompt = f"""Allowed labels: {LABELS}

Retrieved examples:
{demonstrations}

New ticket:
{ticket}
"""
    response = client.responses.create(
        model=TEXT_MODEL,
        reasoning={"effort": "none"},
        instructions=(
            "Use the retrieved examples as demonstrations, then classify the new ticket. "
            "Return exactly one allowed label in the structured result."
        ),
        input=prompt,
        text={"format": LABEL_SCHEMA},
        store=False,
    )
    return json.loads(response.output_text)["label"]


pred_rag_openai = predict_with_cache(
    "few_shot_openai_embeddings",
    lambda ticket: classify_ticket_with_rag(ticket, openai_store),
)
pred_rag_open = predict_with_cache(
    "few_shot_open_embeddings",
    lambda ticket: classify_ticket_with_rag(ticket, open_store),
)

print(f"Few-shot - {OPENAI_STORE_LABEL}")
print(classification_report(test_df["label"], pred_rag_openai, labels=LABELS, zero_division=0))
print("Few-shot - open embeddings")
print(classification_report(test_df["label"], pred_rag_open, labels=LABELS, zero_division=0))

few_shot_openai_embeddings: 10/153
few_shot_openai_embeddings: 20/153
few_shot_openai_embeddings: 30/153
few_shot_openai_embeddings: 40/153
few_shot_openai_embeddings: 50/153
few_shot_openai_embeddings: 60/153
few_shot_openai_embeddings: 70/153
few_shot_openai_embeddings: 80/153
few_shot_openai_embeddings: 90/153
few_shot_openai_embeddings: 100/153
few_shot_openai_embeddings: 110/153
few_shot_openai_embeddings: 120/153
few_shot_openai_embeddings: 130/153
few_shot_openai_embeddings: 140/153
few_shot_openai_embeddings: 150/153
few_shot_openai_embeddings: 153/153
few_shot_open_embeddings: 10/153
few_shot_open_embeddings: 20/153
few_shot_open_embeddings: 30/153
few_shot_open_embeddings: 40/153
few_shot_open_embeddings: 50/153
few_shot_open_embeddings: 60/153
few_shot_open_embeddings: 70/153
few_shot_open_embeddings: 80/153
few_shot_open_embeddings: 90/153
few_shot_open_embeddings: 100/153
few_shot_open_embeddings: 110/153
few_shot_open_embeddings: 120/153
few_shot_open_embeddings: 130/153


In [8]:
#comparing all five classifiers
def metric_row(name, predictions):
    return {
        "classifier": name,
        "accuracy": accuracy_score(test_df["label"], predictions),
        "macro_f1": f1_score(test_df["label"], predictions, average="macro", zero_division=0),
        "weighted_f1": f1_score(test_df["label"], predictions, average="weighted", zero_division=0),
    }


results = pd.DataFrame([
    metric_row("Zero-shot", pred_zero_shot),
    metric_row(f"KNN - {OPENAI_STORE_LABEL}", pred_knn_openai),
    metric_row("KNN - open embeddings", pred_knn_open),
    metric_row(f"Few-shot - {OPENAI_STORE_LABEL}", pred_rag_openai),
    metric_row("Few-shot - open embeddings", pred_rag_open),
]).sort_values("macro_f1", ascending=False).reset_index(drop=True)

display(results.style.format({"accuracy": "{:.3f}", "macro_f1": "{:.3f}", "weighted_f1": "{:.3f}"}))
results.to_csv(ROOT / "outputs" / "part_a" / "classifier_comparison.csv", index=False)

,classifier,accuracy,macro_f1,weighted_f1
0,Few-shot - open embeddings,0.693,0.706,0.703
1,Few-shot - OpenAI embeddings (text-embedding-3-small),0.667,0.675,0.676
2,KNN - OpenAI embeddings (text-embedding-3-small),0.608,0.596,0.614
3,KNN - open embeddings,0.582,0.543,0.572
4,Zero-shot,0.549,0.456,0.508


##Part B: Data extraction


In [9]:
#schema and page renderer
IBES_COLUMNS = ["file_name", "key", "item", "data_type", "format", "length", "start", "end", "comments"]

IBES_SCHEMA = {
    "type": "json_schema",
    "name": "ibes_table_extraction",
    "schema": {
        "type": "object",
        "properties": {
            "data_records": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "file_name": {"type": ["string", "null"]},
                        "key": {"type": ["string", "null"]},
                        "item": {"type": ["string", "null"]},
                        "data_type": {"type": ["string", "null"]},
                        "format": {"type": ["string", "null"]},
                        "length": {"type": ["integer", "null"]},
                        "start": {"type": ["integer", "null"]},
                        "end": {"type": ["integer", "null"]},
                        "comments": {"type": ["string", "null"]},
                    },
                    "required": IBES_COLUMNS,
                    "additionalProperties": False,
                },
            }
        },
        "required": ["data_records"],
        "additionalProperties": False,
    },
    "strict": True,
}


def page_as_data_url(pdf_path, page_number=0, zoom=2):
    with fitz.open(pdf_path) as document:
        page = document[page_number]
        pixmap = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), alpha=False)
    encoded = base64.b64encode(pixmap.tobytes("png")).decode("ascii")
    return "data:image/png;base64," + encoded

In [10]:
#extract, normalize, and save a page
def extract_ibes_page(pdf_path, page_number=0):
    response = client.responses.create(
        model=TEXT_MODEL,
        reasoning={"effort": "none"},
        input=[{
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": (
                        "Extract every actual data row from each visible IBES file-layout table in reading order. "
                        "The visible columns are Key, Item, optional Data Type, Format, Length, Start, End, and Comments. "
                        "Some pages do not have a Data Type column: on those pages set data_type to null and put the "
                        "visible Format value only in format. Repeat the applicable section or file name in file_name "
                        "for every data row, including when a merged heading supplies it. Treat labels such as Basic "
                        "Block and Pricing Block as grouping context, not standalone data rows. Ignore a next-section "
                        "title when its table is not visible. Do extract every row of a partial table that continues "
                        "from a previous page, even when its heading and column labels are above the visible page; use "
                        "null for file_name if that continued table's name is not visible. Preserve visible text exactly "
                        "where possible, use null for blank cells, and never invent a value that is not visible."
                    ),
                },
                {"type": "input_image", "image_url": page_as_data_url(pdf_path, page_number), "detail": "high"},
            ],
        }],
        text={"format": IBES_SCHEMA},
        store=False,
    )
    records = json.loads(response.output_text)["data_records"]
    frame = pd.DataFrame(records, columns=IBES_COLUMNS)
    item_text = frame["item"].astype("string").fillna("").str.strip().str.lower()
    frame = frame.loc[~item_text.isin(["", "na", "nan", "none"])].copy()
    frame["key"] = frame["key"].astype("string").str.replace(r"\s*#\s*", "#", regex=True)
    duplicate_format = (
        frame["data_type"].notna()
        & frame["format"].notna()
        & (frame["data_type"].astype("string").str.strip() == frame["format"].astype("string").str.strip())
    )
    frame.loc[duplicate_format, "data_type"] = pd.NA
    for column in ["length", "start", "end"]:
        frame[column] = pd.to_numeric(frame[column], errors="coerce").astype("Int64")
    return frame.reset_index(drop=True)


def save_extraction(frame, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(output_path, index=False)
    return output_path

In [16]:
#execute and validate the three supplied one-page PDFs
SAMPLE_CASES = [
    ("ibes_detail_history_docs_13.pdf", "ibes_detail_history_docs_13.csv", "ibes_detail_history_docs_13_extracted_v3.csv"),
    ("ibes_detail_history_docs_15.pdf", "ibes_detail_history_docs_15.csv", "ibes_detail_history_docs_15_extracted_v2.csv"),
    ("ibes_summary_history_docs_14.pdf", "ibes_summary_history_docs_14.csv", "ibes_summary_history_docs_14_extracted_v2.csv"),
]

part_b_output = ROOT / "outputs" / "part_b"
validation_rows = []


def meaningful_rows(frame):
    cleaned = frame.copy()
    item_text = cleaned["item"].astype("string").fillna("").str.strip().str.lower()
    return cleaned.loc[~item_text.isin(["", "na", "nan", "none"])].reset_index(drop=True)


def normalized_key_item_pairs(frame):
    keys = (
        frame["key"].astype("string").fillna("").str.strip().str.lower()
        .str.replace(r"\s*#\s*", "#", regex=True)
    )
    items = frame["item"].astype("string").fillna("").str.strip().str.lower()
    return set(zip(keys, items))

for pdf_name, reference_name, output_name in SAMPLE_CASES:
    output_path = part_b_output / output_name
    if output_path.exists():
        extracted = pd.read_csv(output_path)
        print(f"Using cached extraction: {output_path}")
    else:
        extracted = extract_ibes_page(ROOT / pdf_name)
        save_extraction(extracted, output_path)

    reference = pd.read_csv(ROOT / reference_name)
    reference_clean = meaningful_rows(reference)
    extracted_clean = meaningful_rows(extracted)
    reference_pairs = normalized_key_item_pairs(reference_clean)
    extracted_pairs = normalized_key_item_pairs(extracted_clean)
    validation_rows.append({
        "pdf": pdf_name,
        "reference_data_rows": len(reference_clean),
        "extracted_data_rows": len(extracted_clean),
        "row_count_matches": len(reference_clean) == len(extracted_clean),
        "key_item_coverage": len(reference_pairs & extracted_pairs) / max(len(reference_pairs), 1),
        "all_key_items_match": reference_pairs == extracted_pairs,
        "output": str(output_path.relative_to(ROOT)),
    })

validation = pd.DataFrame(validation_rows)
display(validation)
validation.to_csv(part_b_output / "sample_validation_summary_v2.csv", index=False)

,pdf,reference_data_rows,extracted_data_rows,row_count_matches,key_item_coverage,all_key_items_match,output
0,ibes_detail_history_docs_13.pdf,27,27,True,1.0,True,outputs/part_b/ibes_detail_history_docs_13_ext...
1,ibes_detail_history_docs_15.pdf,18,18,True,1.0,True,outputs/part_b/ibes_detail_history_docs_15_ext...
2,ibes_summary_history_docs_14.pdf,22,22,True,1.0,True,outputs/part_b/ibes_summary_history_docs_14_ex...


In [17]:
#full-document extraction pipeline
def extract_full_pdf(pdf_path, output_path, start_page=0):
    with fitz.open(pdf_path) as document:
        page_count = len(document)

    frames = []
    for page_number in range(start_page, page_count):
        page_frame = extract_ibes_page(pdf_path, page_number)
        page_frame.insert(0, "source_page", page_number + 1)
        frames.append(page_frame)
        combined = pd.concat(frames, ignore_index=True)
        save_extraction(combined, output_path)  # checkpoint after each page
        print(f"{pdf_path.name}: {page_number + 1}/{page_count}")
    return pd.concat(frames, ignore_index=True)


full_document_plan = pd.DataFrame([
    {"pdf": "ibes_detail_history_docs.pdf", "pages": len(fitz.open(ROOT / "ibes_detail_history_docs.pdf"))},
    {"pdf": "ibes_summary_history_docs.pdf", "pages": len(fitz.open(ROOT / "ibes_summary_history_docs.pdf"))},
])
display(full_document_plan)


,pdf,pages
0,ibes_detail_history_docs.pdf,55
1,ibes_summary_history_docs.pdf,45
